# Variant Risk Explainer: ClinVar + DNABERT-2 Fine-tuning

Research and education only. This notebook does not produce a clinically validated diagnostic model.

Use GRCh38 for both ClinVar coordinates and reference FASTA sequence extraction.

## 1. Runtime

In Colab, choose `Runtime -> Change runtime type -> GPU` before running the training cells.

In [ ]:
!python --version
!nvidia-smi || true

## 2. Install dependencies

Upload or clone this repository into `/content/variant-risk-explainer`, then install the Colab requirements.

In [ ]:
%cd /content/variant-risk-explainer
!pip install -q -r training/requirements-colab.txt

## 3. Download ClinVar GRCh38 VCF

In [ ]:
!mkdir -p training/data
!wget -q -O training/data/clinvar_grch38.vcf.gz https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz
!ls -lh training/data/clinvar_grch38.vcf.gz

## 4. Provide GRCh38 FASTA

Mount Google Drive or upload a GRCh38 FASTA. The FASTA must match GRCh38 and should be indexed. If the `.fai` index is missing, the next cell creates it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REFERENCE_FASTA = '/content/drive/MyDrive/genomics/GRCh38.primary_assembly.genome.fa.gz'
print(REFERENCE_FASTA)

In [ ]:
import os
if not os.path.exists(REFERENCE_FASTA + '.fai'):
    import pysam
    pysam.faidx(REFERENCE_FASTA)
print('FASTA index ready')

Optional: store the FASTA path in an environment variable for scripts or later notebook cells.

In [ ]:
import os
os.environ['REFERENCE_FASTA_PATH'] = REFERENCE_FASTA

## 5. Prepare ClinVar sequence examples

For a quick smoke test, keep `--max-records 2000`. For a full run, remove it.

In [ ]:
!python training/scripts/prepare_clinvar_dataset.py \
  --clinvar-vcf training/data/clinvar_grch38.vcf.gz \
  --reference-fasta "$REFERENCE_FASTA" \
  --output-jsonl training/data/clinvar_grch38_dnabert2.jsonl \
  --window-size 251 \
  --max-records 2000

## 6. Fine-tune DNABERT-2

Use a small smoke-test run first. Increase epochs, remove `--max-records` during preparation, and tune batch size based on GPU memory for real experiments.

In [ ]:
!python training/scripts/train_dnabert2_classifier.py \
  --dataset-jsonl training/data/clinvar_grch38_dnabert2.jsonl \
  --output-dir training/output/dnabert2-clinvar-grch38 \
  --epochs 1 \
  --batch-size 4 \
  --max-length 256

## 7. Export model artifact

The backend expects a Hugging Face model directory containing tokenizer and model files. Copy or zip `training/output/dnabert2-clinvar-grch38/final_model`.

In [ ]:
!zip -r dnabert2-clinvar-grch38-final-model.zip training/output/dnabert2-clinvar-grch38/final_model
!ls -lh dnabert2-clinvar-grch38-final-model.zip